In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import property_management
import properties

SOURCE_PATH = "/Volumes/property_catalog/property_management/landing_zone/properties.csv"
TARGET_SCHEMA = "property_catalog.property_management"

RAW_TABLE = f"{TARGET_SCHEMA}.raw_properties"
CLEAN_TABLE = f"{TARGET_SCHEMA}.clean_properties"
REJECTED_TABLE = f"{TARGET_SCHEMA}.rejected_properties"

In [0]:
properties_df = property_management.extract_property_management_data(spark, SOURCE_PATH)

# Incrementally load raw property records
property_management.upsert_property_management_table(spark, properties_df, RAW_TABLE, key_column="Id")

display(properties_df)

In [0]:
# Read raw properties Delta table
properties_df = spark.table(RAW_TABLE)

transformed_properties_df = properties.transform_properties(properties_df)

In [0]:
validated_properties_df = properties.validate_properties(transformed_properties_df)

display(validated_properties_df)

In [0]:
clean_properties_df, rejected_properties_df = properties.separate_properties(validated_properties_df)

print("Clean properties:")
display(clean_properties_df)

print("Rejected properties:")
display(rejected_properties_df)

In [0]:
properties_summary = property_management.summarise_property_management_etl(properties_df, clean_properties_df, rejected_properties_df, SOURCE_PATH)

In [0]:
# Remove validation flags from the clean output
clean_properties_output_df = clean_properties_df.drop(
    "invalid_property_id",
    "duplicate_property_id",
    "invalid_rent",
    "invalid_address",
    "invalid_status",
    "invalid_landlord_id"
)

# Incrementally load clean property records
property_management.upsert_property_management_table(spark, clean_properties_output_df, CLEAN_TABLE, key_column="Id")

# Incrementally load rejected property records
property_management.upsert_property_management_table(spark, rejected_properties_df, REJECTED_TABLE, key_column="Id")

print(f"Clean property records saved to: {CLEAN_TABLE}")
print(f"Rejected property records saved to: {REJECTED_TABLE}")